# Statistical Analysis — AG2 Stage 2 Interventions

Paired statistical tests comparing each Stage 2 intervention against the Stage 1 baseline on the same OlympiadBench tasks.

**Statistical methods:**
- **McNemar test** (paired binary, applied to accuracy and every FM): exact binomial when b+c < 25; χ² with continuity correction otherwise
- **Bootstrap 95% CI**: 10 000 paired resamples keyed on `question_id`, seed = 42

In [53]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
from IPython.display import display

# make AG2/paths.py importable (this notebook lives in AG2/analysis/)
sys.path.insert(0, str(Path('..').resolve()))
import paths

## Configuration

In [54]:
RESULTS = paths.RESULTS_DIR

BASELINE = RESULTS / 'baseline_olympiad_gpt41_n50_20260609'

INTERVENTIONS = [
    (
        'v1 - FM-2.6 Action-Reasoning Mismatch',
        RESULTS / 'stage_2_v1_olympiad_gpt41_n50_20260611',
        '2.6',
    ),
    (
        'v2 - FM-1.1 Disobey Task Specification',
        RESULTS / 'stage_2_v2_olympiad_gpt41_n50_20260612',
        '1.1',
    ),
    (
        'v3 - FM-3.3 Weak Verification',
        RESULTS / 'stage_2_v3_olympiad_gpt41_n50_20260613',
        '3.3',
    ),
]

FM_CODES = [
    '1.1', '1.2', '1.3', '1.4', '1.5',
    '2.1', '2.2', '2.3', '2.4', '2.5', '2.6',
    '3.1', '3.2', '3.3',
]

FM_NAMES = {
    '1.1': 'Disobey Task Specification',
    '1.2': 'Disobey Role Specification',
    '1.3': 'Step Repetition',
    '1.4': 'Loss of Conversation History',
    '1.5': 'Unaware of Termination Conditions',
    '2.1': 'Conversation Reset',
    '2.2': 'Fail to Ask for Clarification',
    '2.3': 'Task Derailment',
    '2.4': 'Information Withholding',
    '2.5': "Ignored Other Agent's Input",
    '2.6': 'Action-Reasoning Mismatch',
    '3.1': 'Premature Termination',
    '3.2': 'No or Incorrect Verification',
    '3.3': 'Weak Verification',
}

## Data Loading

In [55]:
def load_run(run_dir: Path, label: str) -> pd.DataFrame:
    pred = pd.read_csv(run_dir / paths.JUDGE_SUBDIR / 'predictions.csv')
    summ = pd.read_csv(run_dir / 'summary.csv')[['trace_id', 'question_id', 'correct']]
    # Robustly parse correct column regardless of bool/string encoding
    summ['correct'] = (
        summ['correct']
        .map({'True': True, 'False': False, True: True, False: False})
        .astype(bool)
    )
    df = pred.merge(summ, on='trace_id', how='left')
    df['_run'] = label
    return df


df_base = load_run(BASELINE, 'Baseline')
print(f'Baseline : {len(df_base):3d} traces  |  accuracy = {df_base["correct"].mean():.3f}')
print()
for label, idir, fm in INTERVENTIONS:
    df_i = load_run(idir, label)
    print(f'{label}')
    print(f'  {len(df_i):3d} traces  |  accuracy = {df_i["correct"].mean():.3f}  |  target FM-{fm} prevalence = {df_i[fm].mean():.3f}')

Baseline :  49 traces  |  accuracy = 0.653

v1 - FM-2.6 Action-Reasoning Mismatch
   49 traces  |  accuracy = 0.633  |  target FM-2.6 prevalence = 0.265
v2 - FM-1.1 Disobey Task Specification
   48 traces  |  accuracy = 0.708  |  target FM-1.1 prevalence = 0.125
v3 - FM-3.3 Weak Verification
   49 traces  |  accuracy = 0.694  |  target FM-3.3 prevalence = 0.184


In [56]:
runs_overview = [
    ('Baseline',      df_base),
    ('Int 1  FM-1.1', load_run(INTERVENTIONS[1][1], 'int1')),
    ('Int 2  FM-3.3', load_run(INTERVENTIONS[2][1], 'int2')),
    ('Int 3  FM-2.6', load_run(INTERVENTIONS[0][1], 'int3')),
]

rows = []
for label, df in runs_overview:
    has_fm = df[FM_CODES].any(axis=1)
    rows.append({
        'Run'              : label,
        'Traces with >= 1 FM': f'{has_fm.sum()}/{len(df)}',
        '%'                : round(has_fm.mean() * 100, 1),
    })

display(
    pd.DataFrame(rows).style
    .hide(axis='index')
    .format({'%': '{:.1f}%'})
    .set_caption('Traces with at least one FM flagged')
    .set_properties(**{'text-align': 'right'}, subset=['Traces with >= 1 FM', '%'])
    .set_properties(**{'text-align': 'left'}, subset=['Run'])
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '1.05em'), ('font-weight', 'bold'), ('text-align', 'left')]},
        {'selector': 'th',
         'props': [('text-align', 'center'), ('background-color', '#f5f5f5')]},
    ])
)

Run,Traces with >= 1 FM,%
Baseline,23/49,46.9%
Int 1 FM-1.1,17/48,35.4%
Int 2 FM-3.3,16/49,32.7%
Int 3 FM-2.6,19/49,38.8%


## Pairing & Statistical Functions

In [57]:
def pair_runs(df_left: pd.DataFrame, df_right: pd.DataFrame) -> pd.DataFrame:
    """Inner join on question_id → one row per shared task."""
    left  = df_left.set_index('question_id')
    right = df_right.set_index('question_id')
    paired = left.merge(
        right, left_index=True, right_index=True, suffixes=('_base', '_int')
    )
    return paired.reset_index()

In [58]:
def mcnemar_test(base_bin: np.ndarray, int_bin: np.ndarray):
    """
    McNemar test for paired binary outcomes.

    For accuracy:  base=1 means baseline correct,  int=1 means intervention correct.
    For FM columns: base=1 means FM present in baseline, int=1 means FM present in intervention.

    Returns (b, c, statistic, p_value, method):
      b = base=1 & int=0
      c = base=0 & int=1
    Uses exact binomial when b+c < 25, chi-square with continuity correction otherwise.
    """
    x = base_bin.astype(int)
    y = int_bin.astype(int)
    b = int(((x == 1) & (y == 0)).sum())
    c = int(((x == 0) & (y == 1)).sum())
    n = b + c
    if n == 0:
        return b, c, np.nan, np.nan, '—'
    if n < 25:
        p = stats.binomtest(min(b, c), n=n, p=0.5, alternative='two-sided').pvalue
        return b, c, float(min(b, c)), float(p), 'exact binomial'
    stat = (abs(b - c) - 1) ** 2 / n
    p = stats.chi2.sf(stat, df=1)
    return b, c, float(stat), float(p), 'χ² (CC)'


def bootstrap_ci(
    base_vals: np.ndarray,
    int_vals: np.ndarray,
    n_boot: int = 10_000,
    seed: int = 42,
):
    """Paired bootstrap 95% CI on mean(intervention) − mean(baseline)."""
    n   = len(base_vals)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_diffs = int_vals[idx].mean(axis=1) - base_vals[idx].mean(axis=1)
    return float(np.percentile(boot_diffs, 2.5)), float(np.percentile(boot_diffs, 97.5))

## Table Building & Display Helpers

In [59]:
def _fmt_p(p: float) -> str:
    if np.isnan(p):
        return '—'
    if p < 0.001:
        return '< 0.001'
    return f'{p:.3f}'


def _fmt_ci(lo: float, hi: float) -> str:
    def _s(v):
        pp = v * 100
        return f'+{pp:.1f}pp' if pp >= 0 else f'{pp:.1f}pp'
    return f'[{_s(lo)}, {_s(hi)}]'


def _fmt_diff(d: float) -> str:
    pp = d * 100
    return f'+{pp:.1f}pp' if pp >= 0 else f'{pp:.1f}pp'


def build_summary_table(int_dir: Path, target_fm: str) -> pd.DataFrame:
    """Return a 15-row summary DataFrame for one intervention vs baseline."""
    df_int = load_run(int_dir, 'Intervention')
    paired = pair_runs(df_base, df_int)
    n_paired = len(paired)

    rows = []

    # ── Accuracy ─────────────────────────────────────────────────────────────
    bv = paired['correct_base'].astype(float).values
    iv = paired['correct_int'].astype(float).values
    b, c, stat, p, method = mcnemar_test(bv, iv)
    ci_lo, ci_hi = bootstrap_ci(bv, iv)
    rows.append({
        'Metric'      : 'Accuracy',
        'Baseline'    : round(float(bv.mean()), 3),
        'Intervention': round(float(iv.mean()), 3),
        'Difference'  : _fmt_diff(iv.mean() - bv.mean()),
        '95% CI'      : _fmt_ci(ci_lo, ci_hi),
        'p-value'     : _fmt_p(p),
        'Method'      : method,
        '_target'     : False,
        '_n_paired'   : n_paired,
    })

    # ── FM rows ───────────────────────────────────────────────────────────────
    for fm in FM_CODES:
        bv = paired[f'{fm}_base'].astype(float).values
        iv = paired[f'{fm}_int'].astype(float).values
        b, c, stat, p, method = mcnemar_test(bv, iv)
        ci_lo, ci_hi = bootstrap_ci(bv, iv)
        rows.append({
            'Metric'      : f'FM {fm}  {FM_NAMES[fm]}',
            'Baseline'    : round(float(bv.mean()), 3),
            'Intervention': round(float(iv.mean()), 3),
            'Difference'  : _fmt_diff(iv.mean() - bv.mean()),
            '95% CI'      : _fmt_ci(ci_lo, ci_hi),
            'p-value'     : _fmt_p(p),
            'Method'      : method,
            '_target'     : fm == target_fm,
            '_n_paired'   : n_paired,
        })

    return pd.DataFrame(rows)


def display_summary(df: pd.DataFrame, title: str) -> None:
    n = int(df['_n_paired'].iloc[0])

    vis_cols = ['Metric', 'Baseline', 'Intervention', 'Δ (pp)', '95% CI', 'p-value']
    df = df.rename(columns={'Difference': 'Δ (pp)'}).copy()

    def _highlight(row):
        if row['_target']:
            return ['background-color: #fff9c4; font-weight: bold'] * len(row)
        return [''] * len(row)

    styled = (
        df[vis_cols + ['_target']]
        .style
        .apply(_highlight, axis=1)
        .hide(axis='index')
        .hide(['_target'], axis='columns')
        .set_caption(f'{title}  (n = {n} paired tasks)')
        .set_properties(
            **{'text-align': 'right'},
            subset=['Baseline', 'Intervention', 'Δ (pp)', '95% CI', 'p-value'],
        )
        .set_properties(**{'text-align': 'left'}, subset=['Metric'])
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '1.05em'), ('font-weight', 'bold'), ('text-align', 'left')]},
            {'selector': 'th',
             'props': [('text-align', 'center'), ('background-color', '#f5f5f5')]},
        ])
    )
    display(styled)

---
## Intervention 1: FM-1.1 Disobey Task Specification

**Prompt addition:** *"Always provide the general solution or expression requested by the task, even if a specific numerical example is used to verify the approach. Final answers must follow the requested output format exactly. Use exact symbolic forms such as fractions or radicals instead of decimal approximations unless the problem explicitly asks for decimals."*

In [60]:
v2_label, v2_dir, v2_fm = INTERVENTIONS[1]
t_v2 = build_summary_table(v2_dir, v2_fm)
display_summary(t_v2, v2_label)

Metric,Baseline,Intervention,Δ (pp),95% CI,p-value
Accuracy,0.667000,0.708000,+4.2pp,"[-4.2pp, +12.5pp]",0.625
FM 1.1 Disobey Task Specification,0.292000,0.125000,-16.7pp,"[-33.3pp, -2.1pp]",0.077
FM 1.2 Disobey Role Specification,0.021000,0.000000,-2.1pp,"[-6.2pp, +0.0pp]",1.000
FM 1.3 Step Repetition,0.083000,0.042000,-4.2pp,"[-14.6pp, +6.2pp]",0.688
FM 1.4 Loss of Conversation History,0.021000,0.021000,+0.0pp,"[-6.2pp, +6.2pp]",1.000
FM 1.5 Unaware of Termination Conditions,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—
FM 2.1 Conversation Reset,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—
FM 2.2 Fail to Ask for Clarification,0.062000,0.021000,-4.2pp,"[-12.5pp, +4.2pp]",0.625
FM 2.3 Task Derailment,0.083000,0.000000,-8.3pp,"[-16.7pp, -2.1pp]",0.125
FM 2.4 Information Withholding,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—


---
## Intervention 2: FM-3.3 Weak Verification

**Prompt addition:** *"After obtaining a result from code, verify the answer by substituting it back into the original problem or checking it against a known constraint before writing TERMINATE."*

In [61]:
v3_label, v3_dir, v3_fm = INTERVENTIONS[2]
t_v3 = build_summary_table(v3_dir, v3_fm)
display_summary(t_v3, v3_label)

Metric,Baseline,Intervention,Δ (pp),95% CI,p-value
Accuracy,0.667000,0.708000,+4.2pp,"[-6.2pp, +14.6pp]",0.688
FM 1.1 Disobey Task Specification,0.271000,0.167000,-10.4pp,"[-25.0pp, +4.2pp]",0.302
FM 1.2 Disobey Role Specification,0.021000,0.000000,-2.1pp,"[-6.2pp, +0.0pp]",1.000
FM 1.3 Step Repetition,0.062000,0.042000,-2.1pp,"[-8.3pp, +4.2pp]",1.000
FM 1.4 Loss of Conversation History,0.021000,0.021000,+0.0pp,"[-6.2pp, +6.2pp]",1.000
FM 1.5 Unaware of Termination Conditions,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—
FM 2.1 Conversation Reset,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—
FM 2.2 Fail to Ask for Clarification,0.062000,0.021000,-4.2pp,"[-12.5pp, +4.2pp]",0.625
FM 2.3 Task Derailment,0.083000,0.000000,-8.3pp,"[-16.7pp, -2.1pp]",0.125
FM 2.4 Information Withholding,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—


---
## Intervention 3: FM-2.6 Action-Reasoning Mismatch

**Prompt addition:** *"Each code block must be completely self-contained: always include all necessary imports and variable definitions, even if they appeared in a previous block. Before writing code, explicitly state what the code will compute and ensure the implementation matches that description exactly."*

In [62]:
v1_label, v1_dir, v1_fm = INTERVENTIONS[0]
t_v1 = build_summary_table(v1_dir, v1_fm)
display_summary(t_v1, v1_label)

Metric,Baseline,Intervention,Δ (pp),95% CI,p-value
Accuracy,0.653000,0.633000,-2.0pp,"[-12.2pp, +8.2pp]",1.000
FM 1.1 Disobey Task Specification,0.286000,0.184000,-10.2pp,"[-24.5pp, +4.1pp]",0.267
FM 1.2 Disobey Role Specification,0.020000,0.020000,+0.0pp,"[-6.1pp, +6.1pp]",1.000
FM 1.3 Step Repetition,0.082000,0.041000,-4.1pp,"[-14.3pp, +6.1pp]",0.688
FM 1.4 Loss of Conversation History,0.020000,0.000000,-2.0pp,"[-6.1pp, +0.0pp]",1.000
FM 1.5 Unaware of Termination Conditions,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—
FM 2.1 Conversation Reset,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—
FM 2.2 Fail to Ask for Clarification,0.061000,0.000000,-6.1pp,"[-14.3pp, +0.0pp]",0.250
FM 2.3 Task Derailment,0.082000,0.000000,-8.2pp,"[-16.3pp, -2.0pp]",0.125
FM 2.4 Information Withholding,0.000000,0.000000,+0.0pp,"[+0.0pp, +0.0pp]",—


---
## Combined overview: all three interventions

Side-by-side comparison of all three interventions.
FMs sorted by mean absolute Δ descending.

In [ ]:
def _parse_diff(s: str) -> float:
    """Parse '+8.2pp' → 0.082 for sorting in the combined table."""
    try:
        return float(s.replace('pp', '')) / 100
    except ValueError:
        return 0.0


def _fmt_pp(v: float) -> str:
    pp = v * 100
    return f'+{pp:.1f}pp' if pp >= 0 else f'{pp:.1f}pp'


def build_combined(table_triples):
    first_df = table_triples[0][2]
    metrics  = first_df['Metric'].str.replace(' ★', '', regex=False).tolist()
    n_rows   = len(metrics)

    delta_cols, numeric, target_col_pairs = [], {}, {}
    data = {'Metric': metrics}

    for i, (label, target_fm, df) in enumerate(table_triples, 1):
        short = f'Int. {i}'
        dc, pc = f'{short} Δ', f'{short} p'
        delta_cols.append(dc)
        nums = [_parse_diff(d) for d in df['Difference'].tolist()]
        numeric[dc] = nums
        data[dc] = [_fmt_pp(v) for v in nums]
        data[pc] = df['p-value'].tolist()
        target_col_pairs[target_fm] = (dc, pc)

    combined = pd.DataFrame(data)

    combined['_mean_abs'] = [
        sum(abs(numeric[dc][i]) for dc in delta_cols) / len(delta_cols)
        for i in range(n_rows)
    ]

    acc = combined.iloc[[0]]
    fms = combined.iloc[1:].copy().sort_values('_mean_abs', ascending=False)

    combined = (
        pd.concat([acc, fms])
        .reset_index(drop=True)
        .drop(columns=['_mean_abs'])
    )
    return combined, target_col_pairs


combined, target_cols = build_combined([
    (v2_label, v2_fm, t_v2),  # Int. 1 — FM-1.1
    (v3_label, v3_fm, t_v3),  # Int. 2 — FM-3.3
    (v1_label, v1_fm, t_v1),  # Int. 3 — FM-2.6
])


def _hl_combined(row):
    styles = [''] * len(row)
    col_list = list(row.index)
    for fm, (dc, pc) in target_cols.items():
        if f'FM {fm}' in row['Metric']:
            for col in ('Metric', dc, pc):
                if col in col_list:
                    styles[col_list.index(col)] = 'font-weight: bold'
    return styles


styled_combined = (
    combined.style
    .apply(_hl_combined, axis=1)
    .hide(axis='index')
    .set_caption('Combined Overview — Δ (pp) and p-value for all three interventions vs baseline')
    .set_properties(**{'text-align': 'right'}, subset=combined.columns[1:].tolist())
    .set_properties(**{'text-align': 'left'}, subset=['Metric'])
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '1.05em'), ('font-weight', 'bold'), ('text-align', 'left')]},
        {'selector': 'th',
         'props': [('text-align', 'center'), ('background-color', '#f5f5f5')]},
    ])
)
display(styled_combined)

In [64]:
# 2x2 Contingency Tables – Target FM Prevalence (McNemar)

int_configs = [
    ("Int 1  FM-1.1  Disobey Task Specification", INTERVENTIONS[1][1], '1.1'),
    ("Int 2  FM-3.3  Weak Verification",          INTERVENTIONS[2][1], '3.3'),
    ("Int 3  FM-2.6  Action-Reasoning Mismatch",  INTERVENTIONS[0][1], '2.6'),
]

rows = []
for label, idir, fm in int_configs:
    df_int = load_run(idir, 'Intervention')
    paired = pair_runs(df_base, df_int)
    bv = paired[f'{fm}_base'].astype(int).values
    iv = paired[f'{fm}_int'].astype(int).values
    a = int(((bv == 1) & (iv == 1)).sum())
    b = int(((bv == 1) & (iv == 0)).sum())
    c = int(((bv == 0) & (iv == 1)).sum())
    d = int(((bv == 0) & (iv == 0)).sum())
    rows.append({
        'Intervention': label,
        'a (FM in both)': a,
        'b (baseline only)': b,
        'c (intervention only)': c,
        'd (FM in neither)': d,
    })

ct_df = pd.DataFrame(rows)
display(
    ct_df.style
    .hide(axis='index')
    .set_caption(
        '2x2 Contingency Tables: Target FM Prevalence  '
    )
    .set_properties(**{'text-align': 'right'},
                    subset=['a (FM in both)', 'b (baseline only)', 'c (intervention only)', 'd (FM in neither)'])
    .set_properties(**{'text-align': 'left'}, subset=['Intervention'])
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '1.05em'), ('font-weight', 'bold'), ('text-align', 'left')]},
        {'selector': 'th',
         'props': [('text-align', 'center'), ('background-color', '#f5f5f5')]},
    ])
)

Intervention,a (FM in both),b (baseline only),c (intervention only),d (FM in neither)
Int 1 FM-1.1 Disobey Task Specification,2,12,4,30
Int 2 FM-3.3 Weak Verification,4,5,5,34
Int 3 FM-2.6 Action-Reasoning Mismatch,3,6,10,30


In [65]:
# McNemar Power Analysis – Target FM Prevalence

import numpy as np
from scipy import stats

def cohens_h(p, p0=0.5):
    return 2 * np.arcsin(np.sqrt(p)) - 2 * np.arcsin(np.sqrt(p0))

ALPHA  = 0.05
TARGET = 0.80

z_crit = stats.norm.ppf(1 - ALPHA / 2)
z_beta = stats.norm.ppf(TARGET)

int_configs_power = [
    ("Int 1  FM-1.1  Disobey Task Specification", INTERVENTIONS[1][1], '1.1'),
    ("Int 2  FM-3.3  Weak Verification",          INTERVENTIONS[2][1], '3.3'),
    ("Int 3  FM-2.6  Action-Reasoning Mismatch",  INTERVENTIONS[0][1], '2.6'),
]

rows = []
for label, idir, fm in int_configs_power:
    df_int   = load_run(idir, 'Intervention')
    paired   = pair_runs(df_base, df_int)
    n_paired = len(paired)
    bv = paired[f'{fm}_base'].astype(int).values
    iv = paired[f'{fm}_int'].astype(int).values
    b = int(((bv == 1) & (iv == 0)).sum())
    c = int(((bv == 0) & (iv == 1)).sum())
    n_disc    = b + c
    p         = b / n_disc if n_disc > 0 else 0.5
    abs_h     = abs(cohens_h(p))
    wrong_dir = c > b

    if abs_h < 1e-9:
        current_power = ALPHA
        n_req_str     = "inf"
        n_total_str   = "inf"
    elif wrong_dir:
        ncp           = abs_h * np.sqrt(n_disc)
        current_power = stats.norm.cdf(ncp - z_crit) + stats.norm.cdf(-ncp - z_crit)
        n_req_str     = "n/a"
        n_total_str   = "n/a"
    else:
        ncp           = abs_h * np.sqrt(n_disc)
        current_power = stats.norm.cdf(ncp - z_crit) + stats.norm.cdf(-ncp - z_crit)
        n_req_exact   = ((z_crit + z_beta) / abs_h) ** 2
        disc_rate     = n_disc / n_paired
        n_req_str     = str(int(np.ceil(n_req_exact)))
        n_total_str   = str(int(np.ceil(n_req_exact / disc_rate)))

    rows.append({
        'Intervention'                    : label,
        'b'                               : b,
        'c'                               : c,
        'b+c'                             : n_disc,
        'n paired'                        : n_paired,
        'ratio b/(b+c)'                   : round(p, 3),
        'Current power'                   : round(current_power, 3),
        'Required discordant pairs (80%)' : n_req_str,
        'Required n (80%)'                : n_total_str,
    })

power_df = pd.DataFrame(rows)

display(
    power_df.style
    .hide(axis='index')
    .set_caption(
        'McNemar Power Analysis: Target FM Prevalence  '
    )
    .format({'ratio b/(b+c)': '{:.3f}', 'Current power': '{:.3f}'})
    .set_properties(**{'text-align': 'right'},
                    subset=['b', 'c', 'b+c', 'n paired', 'ratio b/(b+c)', 'Current power',
                            'Required discordant pairs (80%)', 'Required n (80%)'])
    .set_properties(**{'text-align': 'left'}, subset=['Intervention'])
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '1.05em'), ('font-weight', 'bold'), ('text-align', 'left')]},
        {'selector': 'th',
         'props': [('text-align', 'center'), ('background-color', '#f5f5f5')]},
    ])
)

Intervention,b,c,b+c,n paired,ratio b/(b+c),Current power,Required discordant pairs (80%),Required n (80%)
Int 1 FM-1.1 Disobey Task Specification,12,4,16,48,0.750,0.553,29,86
Int 2 FM-3.3 Weak Verification,5,5,10,48,0.500,0.050,inf,inf
Int 3 FM-2.6 Action-Reasoning Mismatch,6,10,16,49,0.375,0.173,n/a,n/a
